# A second judge, from another family

Reviewers 2 and 3 both ask for a judge from a different model family. This runs
Falcon3-7B-Instruct over the same 200 trajectories, with the same prompt, the same
ground-truth block, the same greedy decoding and the same scoring convention as the
Qwen 2.5 7B judge the paper reports at kappa = 0.115.

Nothing is reimplemented: the prompt builder, the reply parser and the lenient kappa
are fetched from the released repository, so the only thing that changes between the
published result and this one is the model.

The scale is held at 7B deliberately, so that the comparison speaks to family rather
than size. Two families of the same size are not of the same capability, so this is a
second configuration and not a controlled experiment isolating family, and the write-up
has to say so.

In [ ]:
import json, re, sys, time, urllib.request
import torch

BASE = ("https://raw.githubusercontent.com/achenna-source/"
        "icv-agent-observability/neucom-d-26-17269-r1/")

def fetch(p):
    return urllib.request.urlopen(BASE + p, timeout=90).read().decode("utf-8")

ns = {}
exec(compile(fetch("code/judge.py"), "judge.py", "exec"), ns)
build_prompt, parse_reply = ns["build_prompt"], ns["parse_reply"]
GEN = ns["GENERATION_KWARGS"]
mns = {}
exec(compile(fetch("code/metrics.py"), "metrics.py", "exec"), mns)
kappa_lenient, cluster_bootstrap = mns["kappa_lenient"], mns["cluster_bootstrap"]
print("judge prompt and metric loaded from the released tag")
print("generation kwargs:", GEN)

rows = [json.loads(l) for l in fetch("data/trajectories_200.jsonl").splitlines() if l.strip()]
_kbfile = json.loads(fetch("data/knowledge_base.json"))
# Le fichier est une enveloppe : les faits sont sous "facts", et les villes dont la
# population est volontairement absente sous "populations_absent_by_design". Les lire
# a plat donnait un bloc de verite-terrain vide, donc un juge aveugle.
kb = _kbfile["facts"]
cities_absent = _kbfile["populations_absent_by_design"]
print("trajectories: %d | facts: %d | cities absent by design: %d"
      % (len(rows), len(kb), len(cities_absent)))
print("cuda devices:", torch.cuda.device_count())

In [ ]:
import ast

def steps_of(r):
    t = r["trajectory"]
    return ast.literal_eval(t) if isinstance(t, str) else t


rec0 = {"task": rows[0]["task"], "trajectory": steps_of(rows[0])}
demo = build_prompt(rec0, kb, cities_absent)
print("\n--- prompt as the published builder renders it, first 1200 chars ---\n")
print(demo[:1200])

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "tiiuae/Falcon3-7B-Instruct"
t0 = time.time()
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16,
                                             device_map="auto")
model.eval()
print("%s loaded in %.0f s" % (MODEL, time.time() - t0))
print("device map spread over %d devices" % torch.cuda.device_count())

In [ ]:
def judge(prompt):
    msgs = [{"role": "user", "content": prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, pad_token_id=tok.eos_token_id, **GEN)
    return tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)

preds, raw = [], []
t0 = time.time()
for i, r in enumerate(rows):
    rec = {"task": r["task"], "trajectory": steps_of(r)}
    reply = judge(build_prompt(rec, kb, cities_absent))
    raw.append(reply)
    preds.append(parse_reply(reply))
    if (i + 1) % 25 == 0:
        print("%3d/%d  %.0f s elapsed" % (i + 1, len(rows), time.time() - t0))
print("done in %.0f s" % (time.time() - t0))

unparsed = sum(1 for p in preds if p is None)
print("replies with no parseable code: %d" % unparsed)
json.dump({"model": MODEL, "preds": preds, "raw": raw},
          open("judge_falcon3_7b.json", "w"), ensure_ascii=False)

In [ ]:
from collections import Counter

human = [r["label_human"] for r in rows]
valid = set(human)
pred = [p if p else "UNPARSED" for p in preds]

agree = sum(1 for a, b in zip(human, pred) if a == b)
k = kappa_lenient(human, pred, valid)
print("Falcon3-7B-Instruct against the first annotator's labels, 200 trajectories")
print("   raw agreement %d/200 = %.1f%%" % (agree, 100.0 * agree / 200))
print("   Cohen's kappa (lenient convention) %.3f" % k)
print("   published Qwen 2.5 7B result for comparison: 0.115")
print()
print("distribution of the codes it assigns: %s" % dict(Counter(pred).most_common()))
print("distribution of the human labels     : %s" % dict(Counter(human).most_common()))

In [ ]:
clusters = [r.get("cluster_id", r["id"]) for r in rows]
items = [(c, (h, p)) for c, h, p in zip(clusters, human, pred)]
lo, hi, _ = cluster_bootstrap(
    items, lambda draw: kappa_lenient([h for h, _ in draw], [p for _, p in draw], valid))
print("cluster bootstrap 95%% interval: [%.3f, %.3f]" % (lo, hi))
print()
print("per mode, matched against the first annotator")
for m in sorted(set(human)):
    idx = [i for i, h in enumerate(human) if h == m]
    hit = sum(1 for i in idx if pred[i] == m)
    print("   %-3s %3d human, matched %3d  (%.0f%%)"
          % (m, len(idx), hit, 100.0 * hit / max(1, len(idx))))
print()
print("=== for the manuscript ===")
print("Falcon3-7B-Instruct, same prompt, same rules, same 200 trajectories")
print("raw agreement %.1f%%, kappa %.3f, interval [%.3f, %.3f]"
      % (100.0 * agree / 200, k, lo, hi))
print("Qwen 2.5 7B, published: raw 30.0%, kappa 0.115")